# Interactive embedding renderer

Explore cached MNIST-family and arXiv layouts with Datashader. This notebook does not load high-dimensional data or refit embeddings. Run notebook 00 to create the MNIST cache; run `data/arxiv/prepare_arxiv_dataset.ipynb` and notebook 03 to prepare the arXiv vectors and coordinates.

In [1]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
sys.path.insert(0, str(PROJECT_ROOT))
from imste.visualization import render_datashader_embedding


## Load available coordinate caches

In [2]:
DATASETS = {}
DATASET_META = {}
mnist_path = PROJECT_ROOT / "data" / "embeddings" / "mnist_gallery_embeddings.npz"
if mnist_path.is_file():
    with np.load(mnist_path, allow_pickle=False) as saved:
        sample_count = int(saved["max_samples"].item()) if "max_samples" in saved else 60_000
        sample_seed = int(saved["random_state"].item()) if "random_state" in saved else 42
        for index, name in enumerate(saved["dataset_names"].astype(str)):
            DATASETS[name] = {
                "labels": saved[f"labels_{index}"],
                "UMAP": (saved[f"umap_{index}"], float(saved[f"umap_seconds_{index}"])),
                "IMSTE": (saved[f"imste_{index}"], float(saved[f"imste_seconds_{index}"])),
            }
            DATASET_META[name] = {
                "source": "mnist",
                "max_samples": sample_count,
                "sample_seed": sample_seed,
            }

arxiv_path = PROJECT_ROOT / "data" / "arxiv" / "cache" / "arxiv_embeddings.npz"
if arxiv_path.is_file():
    try:
        with np.load(arxiv_path, allow_pickle=False) as saved:
            labels = saved["labels"].astype(str)
            umap_embedding = saved["umap"]
            imste_embedding = saved["imste_multilevel"]
            umap_seconds = float(saved["umap_seconds"])
            imste_seconds = float(saved["imste_seconds"])
    except ValueError as error:
        if "Object arrays cannot be loaded" not in str(error):
            raise
        # Older notebook runs saved pandas string labels as object arrays.
        # This project-created cache is local; new caches use Unicode labels.
        with np.load(arxiv_path, allow_pickle=True) as saved:
            labels = saved["labels"].astype(str)
            umap_embedding = saved["umap"]
            imste_embedding = saved["imste_multilevel"]
            umap_seconds = float(saved["umap_seconds"])
            imste_seconds = float(saved["imste_seconds"])
    arxiv_name = f"arXiv ({len(labels):,})"
    DATASETS[arxiv_name] = {
        "labels": labels,
        "UMAP": (umap_embedding, umap_seconds),
        "IMSTE · multilevel initialization": (imste_embedding, imste_seconds),
    }
    DATASET_META[arxiv_name] = {"source": "arxiv"}

if not DATASETS:
    raise FileNotFoundError("No embedding caches found. Run notebook 00 for MNIST or data/arxiv/prepare_arxiv_dataset.ipynb and notebook 03 for arXiv.")
print("Available datasets:", ", ".join(DATASETS))


Available datasets: MNIST (28×28), Fashion-MNIST (28×28), Kuzushiji-MNIST (28×28)


## Explore and refit embeddings

Choose a dataset and layout, adjust the raster size and appearance, or open the IMSTE settings panel to change fit parameters and refit. Refits are kept in notebook memory for comparison; they do not overwrite the source caches.


In [3]:
dataset_widget = widgets.Dropdown(options=list(DATASETS), description="Dataset:", layout=widgets.Layout(width="450px"))
method_widget = widgets.Dropdown(description="Layout:")
width_widget = widgets.IntSlider(value=700, min=400, max=2200, step=100, description="Width:", continuous_update=False)
height_widget = widgets.IntSlider(value=700, min=400, max=1800, step=100, description="Height:", continuous_update=False)
spread_widget = widgets.IntSlider(value=0, min=0, max=12, step=1, description="Spread px:", continuous_update=False)
alpha_widget = widgets.IntSlider(value=40, min=0, max=255, step=5, description="Min alpha:", continuous_update=False)
background_widget = widgets.Dropdown(options=["white", "black"], value="white", description="Background:")
render_button = widgets.Button(description="Render", button_style="primary", icon="image")
refit_button = widgets.Button(description="Refit IMSTE", button_style="warning", icon="refresh")
output = widgets.Output()
refit_output = widgets.Output()

# All estimator options that affect a fit are exposed below. For n_components > 2,
# the viewer displays the first two coordinates. Callback hooks are programmatic
# callbacks, not notebook fit settings. init_coordinates is a cached-layout warm start.
imste_widgets = {
    "n_components": widgets.IntSlider(value=2, min=2, max=8, step=1, description="Dimensions:", continuous_update=False),
    "n_msts": widgets.IntSlider(value=4, min=1, max=64, step=1, description="MST count:", continuous_update=False),
    "n_epochs": widgets.IntSlider(value=500, min=0, max=2000, step=50, description="Epochs:", continuous_update=False),
    "batch_size": widgets.IntSlider(value=32768, min=4096, max=131072, step=4096, description="Batch size:", continuous_update=False),
    "learning_rate": widgets.FloatSlider(value=0.05, min=0.001, max=0.2, step=0.001, readout_format=".3f", description="Learning rate:", continuous_update=False),
    "negative_ratio": widgets.IntSlider(value=1, min=0, max=30, step=1, description="Negative ratio:", continuous_update=False),
    "lambda_rep": widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, readout_format=".2f", description="Repulsion share (λ):", continuous_update=False),
    "random_state": widgets.IntSlider(value=42, min=0, max=999, step=1, description="Random seed:", continuous_update=False),
    "epsilon": widgets.FloatLogSlider(value=1e-4, base=10, min=-6, max=-2, step=0.1, readout_format=".1e", description="Epsilon:", continuous_update=False),
    "device": widgets.Dropdown(options=["auto", "cpu", "mps", "cuda"], value="mps", description="Device:"),
    "logistic_margin": widgets.FloatSlider(value=1.0, min=0.0, max=25.0, step=0.1, description="Logistic margin:", continuous_update=False),
    "logistic_temperature": widgets.FloatSlider(value=0.5, min=0.01, max=5.0, step=0.01, description="Logistic temperature:", continuous_update=False),
    "mst_method": widgets.Dropdown(options=["famst", "prim"], value="famst", description="MST builder:"),
    "mst_neighbors": widgets.IntSlider(value=15, min=1, max=256, step=1, description="MST neighbors:", continuous_update=False),
    "mst_inter_component_edges": widgets.IntSlider(value=5, min=1, max=50, step=1, description="Bridge edges:", continuous_update=False),
    "mst_max_neighbors": widgets.IntSlider(value=15, min=1, max=512, step=1, description="FAMST max neighbors:", continuous_update=False),
    "mst_representatives_per_component": widgets.IntSlider(value=10, min=1, max=100, step=1, description="MST reps / component:", continuous_update=False),
    "pca_init": widgets.Checkbox(value=True, description="PCA initialization"),
    "pca_init_scaling": widgets.Dropdown(options=["global", "none"], value="global", description="PCA scaling:"),
    "multilevel_init": widgets.Checkbox(value=True, description="Multilevel initialization"),
    "multilevel_reduction_ratio": widgets.IntSlider(value=4, min=2, max=16, step=1, description="Multilevel reduction:", continuous_update=False),
    "multilevel_base_size": widgets.IntSlider(value=1024, min=256, max=8192, step=256, description="Multilevel base size:", continuous_update=False),
    "multilevel_epochs": widgets.IntSlider(value=10, min=0, max=100, step=1, description="Multilevel epochs:", continuous_update=False),
    "metric": widgets.Dropdown(options=["euclidean", "cosine"], value="euclidean", description="Metric:"),
    "use_cached_init": widgets.Checkbox(value=False, description="Use cached IMSTE as initial coordinates"),
}
def _parameter_rows(names):
    items = []
    for name in names:
        item = imste_widgets[name]
        item.style.description_width = "190px"
        item.layout = widgets.Layout(width="680px")
        items.append(item)
    return items

parameter_panels = widgets.Accordion(children=[
    widgets.VBox(_parameter_rows([
        "n_components", "n_epochs", "batch_size", "learning_rate", "random_state", "device",
    ])),
    widgets.VBox(_parameter_rows([
        "metric", "n_msts", "mst_method", "mst_neighbors", "mst_max_neighbors",
        "mst_inter_component_edges", "mst_representatives_per_component",
    ])),
    widgets.VBox(_parameter_rows([
        "negative_ratio", "lambda_rep", "epsilon", "logistic_margin", "logistic_temperature",
    ])),
    widgets.VBox(_parameter_rows([
        "pca_init", "pca_init_scaling", "multilevel_init", "multilevel_reduction_ratio",
        "multilevel_base_size", "multilevel_epochs", "use_cached_init",
    ])),
])
parameter_panels.layout = widgets.Layout(width="720px")
for index, title in enumerate(["Embedding and device", "MST graph", "Optimization", "Initialization"]):
    parameter_panels.set_title(index, title)
parameter_note = widgets.HTML(
    "<small>For dimensions above 2, the viewer shows the first two coordinates. "
    "Progress callbacks are Python hooks. Refit results stay in memory and do not "
    "replace cache files.</small>"
)
refit_panel = widgets.VBox([
    widgets.HTML("<span style=\"font-size:10px\">IMSTE fit settings</span>"), parameter_panels,
    parameter_note, refit_button, refit_output,
], layout=widgets.Layout(width="740px"))
fit_input_cache = {}


def update_methods(*_):
    methods = [key for key in DATASETS[dataset_widget.value] if key != "labels"]
    method_widget.options = methods
    method_widget.value = methods[0]


def render_selected(_=None):
    selected = DATASETS[dataset_widget.value]
    embedding, seconds = selected[method_widget.value]
    visible_coordinates = embedding[:, :2]
    image = render_datashader_embedding(
        visible_coordinates,
        selected["labels"],
        width=width_widget.value,
        height=height_widget.value,
        spread_px=spread_widget.value,
        min_alpha=alpha_widget.value,
        background=background_widget.value,
    )
    with output:
        clear_output(wait=True)
        figure_width = 4
        figure_height = figure_width * height_widget.value / width_widget.value
        fig, ax = plt.subplots(figsize=(figure_width, figure_height), constrained_layout=True)
        ax.imshow(image.to_pil())
        runtime = f"{seconds / 60:.1f} min" if seconds >= 60 else f"{seconds:.1f} s"
        component_note = f" (first 2 of {embedding.shape[1]})" if embedding.shape[1] > 2 else ""
        ax.set_title(
            f"{dataset_widget.value} — {method_widget.value}{component_note} (fit: {runtime})",
            fontsize=6,
        )
        ax.set_axis_off()
        display(fig)
        plt.close(fig)


def _load_refit_input(dataset_name):
    if dataset_name in fit_input_cache:
        return fit_input_cache[dataset_name]

    meta = DATASET_META[dataset_name]
    if meta["source"] == "arxiv":
        vector_path = PROJECT_ROOT / "data" / "arxiv" / "cache" / "arxiv_vectors.npy"
        if not vector_path.is_file():
            raise FileNotFoundError(f"Prepared arXiv vectors are missing: {vector_path}")
        X = np.load(vector_path, mmap_mode="r")
        labels = DATASETS[dataset_name]["labels"]
    else:
        from high_dim_mst_gallery import _load_kmnist, _load_openml, _stratified_sample
        from sklearn.preprocessing import StandardScaler

        if dataset_name == "MNIST (28×28)":
            raw_X, raw_labels = _load_openml("mnist_784")
        elif dataset_name == "Fashion-MNIST (28×28)":
            raw_X, raw_labels = _load_openml("Fashion-MNIST")
        elif dataset_name == "Kuzushiji-MNIST (28×28)":
            raw_X, raw_labels = _load_kmnist()
        else:
            raise ValueError(f"No high-dimensional source is configured for {dataset_name!r}.")
        X, labels = _stratified_sample(
            raw_X,
            raw_labels,
            max_samples=meta["max_samples"],
            random_state=meta["sample_seed"],
        )
        X = StandardScaler().fit_transform(np.asarray(X, dtype=np.float32))

    if len(X) != len(DATASETS[dataset_name]["labels"]):
        raise ValueError("Source data and cached layouts have different row counts.")
    fit_input_cache[dataset_name] = (X, labels)
    return fit_input_cache[dataset_name]


def refit_selected_imste(_=None):
    with refit_output:
        clear_output(wait=True)
        dataset_name = dataset_widget.value
        print(f"Preparing input for {dataset_name}…", flush=True)
        try:
            X, fit_labels = _load_refit_input(dataset_name)
            if len(fit_labels) != len(DATASETS[dataset_name]["labels"]):
                raise ValueError("Refit labels do not align with the cached layouts.")
            from imste import IteratedMinimumSpanningTreeEmbedder
            import time

            settings = {name: widget.value for name, widget in imste_widgets.items() if name != "use_cached_init"}
            settings["init_coordinates"] = None
            if imste_widgets["use_cached_init"].value:
                if settings["n_components"] != 2:
                    raise ValueError("Cached coordinates can initialize a 2D refit only.")
                cached_names = [key for key in DATASETS[dataset_name] if key.startswith("IMSTE")]
                if not cached_names:
                    raise ValueError("No cached IMSTE layout is available for initialization.")
                settings["init_coordinates"] = DATASETS[dataset_name][cached_names[0]][0]
            print(f"Fitting IMSTE with {len(X):,} samples × {X.shape[1]:,} features…", flush=True)
            started = time.perf_counter()
            refit_embedding = IteratedMinimumSpanningTreeEmbedder(**settings).fit_transform(X)
            seconds = time.perf_counter() - started
            result_name = f"IMSTE · refit ({seconds:.1f}s)"
            DATASETS[dataset_name][result_name] = (refit_embedding, seconds)
            previous_name = result_name
            update_methods()
            method_widget.value = previous_name
            print(f"Finished in {seconds / 60:.1f} min" if seconds >= 60 else f"Finished in {seconds:.1f} s")
            render_selected()
        except Exception as error:
            print(f"Refit failed: {type(error).__name__}: {error}")


dataset_widget.observe(update_methods, names="value")
render_button.on_click(render_selected)
refit_button.on_click(refit_selected_imste)
update_methods()
display(widgets.VBox([
    dataset_widget, method_widget,
    widgets.HBox([width_widget, height_widget]),
    widgets.HBox([spread_widget, alpha_widget]),
    widgets.HBox([background_widget, render_button]),
    refit_panel,
    output,
]))
render_selected()
